In [ ]:
import os
os.environ["GROQ_API_KEY"] =  "API_KEY_VALUE"

In [ ]:
!pip install -q langchain-groq langchain langchain-community langchain-core requests duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

In [ ]:
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.3 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=2fbc3ebf72a5f9bc3fb031a39d8dd4cc&query={city}'

  response = requests.get(url)

  return response.json()

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
import langchain
print(langchain.__version__)


1.2.0


In [ ]:
from langchain.agents import create_tool_calling_agent
from langchain_classic import hub


ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

In [ ]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt


In [ ]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_tool_calling_agent(
    llm,
    tools=[search_tool, get_weather_data],
)

In [ ]:
from langchain_classic.agents import AgentExecutor

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    return_only_outputs=True,
    handle_parsing_errors=True   # VERY IMPORTANT for Groq
)


In [ ]:
result = agent_executor.invoke({
    "input": "Find the capital of Madhya Pradesh, then find its current weather condition"
})

print(result["output"])




> Entering new AgentExecutor chain...


AttributeError: 'str' object has no attribute 'tool'

In [ ]:
response

{'messages': [AIMessage(content='Question: What is the current weather in New York City?\nThought: I need to know the current weather in New York City.\nAction: get_weather_data\nAction Input: {"city": "New York City"}\nObservation: The function call is made.\nFinal Answer: <function=get_weather_data>{"city": "New York City"}</function>', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 456, 'total_tokens': 528, 'completion_time': 0.121902533, 'completion_tokens_details': None, 'prompt_time': 0.032616314, 'prompt_tokens_details': None, 'queue_time': 0.035910724, 'total_time': 0.154518847}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bada0-ccff-7160-9ace-7032ff3c7400-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 456, 'output_tokens': 72, 'total_tokens': 528})]}